In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Build the landing path once and reuse it for all file formats
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

def path_exists(path):
    """Return True if the path exists, False if it or any parent directory is missing."""
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as e:
        err = str(e).lower()
        if any(s in err for s in ["filenotfoundexception", "no such file", "does not exist", "not found"]):
            return False
        raise

if not path_exists(landing_path):
    msg = f"No new files received at {landing_path}. Exiting notebook."
    print(msg)
    dbutils.notebook.exit(msg)

In [0]:

if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:

from pyspark.sql import functions as F

updated_df = landing_df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())


In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")